In [0]:
%run "../../utils/00_config"

In [0]:
from pyspark.sql import functions as F

# Load source tables
price_history = spark.table(
    get_gold_table("marts", "mart_item_price_history")
)

dim_execution = spark.table(
    get_table("intermediate", "dim_execution")
)

print(f"Price history records: {price_history.count():,}")
print(f"Execution records: {dim_execution.count():,}")

In [0]:
# Prepare execution metadata
execution_metadata = (
    dim_execution
    .select(
        "execution_id",
        F.col("started_at").alias("execution_started_at"),
        F.col("finished_at").alias("execution_finished_at"),
        "execution_duration_seconds",
        F.col("status").alias("execution_status"),
        F.col("response_count").alias("execution_response_count"),
        F.col("item_count").alias("execution_item_count"),
        F.col("total_orders").alias("execution_total_orders")
    )
)

In [0]:
# Validate execution metadata grain
duplicate_executions = (
    execution_metadata
    .groupBy("execution_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

if duplicate_executions > 0:
    raise ValueError(
        f"Execution metadata contains {duplicate_executions} duplicated execution IDs"
    )

# Enrich offer history with execution metadata
rel_item_price_history = (
    price_history
    .join(
        execution_metadata,
        on="execution_id",
        how="left"
    )
)

print(f"Reporting records: {rel_item_price_history.count():,}")

In [0]:
display(
    rel_item_price_history
    .orderBy(
        "item_slug",
        "observed_at",
        "order_type",
        "platinum"
    )
    .limit(100)
)

In [0]:
# Validate offer history grain
display(
    rel_item_price_history.agg(
        F.count("*").alias("total_records"),
        F.countDistinct("execution_id", "order_id").alias("unique_execution_orders"),
        F.countDistinct("execution_id").alias("unique_executions"),
        F.countDistinct("item_slug").alias("unique_items"),
        F.sum(F.when(F.col("order_type") == "buy", 1).otherwise(0)).alias("buy_offers"),
        F.sum(F.when(F.col("order_type") == "sell", 1).otherwise(0)).alias("sell_offers")
    )
)

# Check duplicated offers
duplicates = (
    rel_item_price_history
    .groupBy("execution_id", "order_id")
    .count()
    .filter(F.col("count") > 1)
)

print(f"Duplicated execution orders: {duplicates.count():,}")

# Check missing execution metadata
missing_metadata = (
    rel_item_price_history
    .filter(
        F.col("execution_started_at").isNull()
        | F.col("execution_status").isNull()
    )
    .count()
)

print(f"Records with missing execution metadata: {missing_metadata:,}")

# Check missing offer identifiers
missing_order_ids = (
    rel_item_price_history
    .filter(F.col("order_id").isNull())
    .count()
)

print(f"Records with missing order IDs: {missing_order_ids:,}")

if duplicates.count() > 0:
    raise ValueError("Duplicate execution-order combinations detected")

if missing_metadata > 0:
    raise ValueError("Missing execution metadata detected")

if missing_order_ids > 0:
    raise ValueError("Missing order IDs detected")

In [0]:
# Validate order type distribution across source and reporting tables
print("MART ITEM PRICE HISTORY")

display(
    price_history
    .groupBy("order_type")
    .count()
    .orderBy("order_type")
)

print("REPORTING ITEM PRICE HISTORY")

display(
    rel_item_price_history
    .groupBy("order_type")
    .count()
    .orderBy("order_type")
)

In [0]:
# Validate Buy and Sell offer counts against source
print("SOURCE: MART ITEM PRICE SUMMARY")

display(
    market_summary
    .groupBy("order_type")
    .agg(
        F.sum("total_offers").alias("total_offers")
    )
    .orderBy("order_type")
)

print("REPORTING: MARKET OVERVIEW")

display(
    rel_market_overview
    .agg(
        F.sum("buy_total_offers").alias("buy_offers"),
        F.sum("sell_total_offers").alias("sell_offers"),
        F.sum("total_market_offers").alias("total_offers")
    )
)

In [0]:
# Define target table
TARGET_TABLE = get_gold_table(
    "reporting",
    "rel_item_price_history"
)

# Materialize reporting table
(
    rel_item_price_history
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(TARGET_TABLE)
)

print(f"Table successfully saved: {TARGET_TABLE}")

In [0]:
# Validate materialized table
saved_history = spark.table(TARGET_TABLE)

print(f"Saved records: {saved_history.count():,}")

display(
    saved_history
    .orderBy("item_slug", "observed_at", "order_type", "platinum")
    .limit(100)
)